In [ ]:
!apt-get update
!apt-get install g++ openjdk-8-jdk
!pip3 install konlpy
!pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0+cu113 torchtext==0.6.0 -f https://download.pytorch.org/whl/torch_stable.html

In [ ]:
import urllib.request
import pandas as pd

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
                           filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
                           filename="ratings_test.txt")

('ratings_test.txt', <http.client.HTTPMessage at 0x7a76279860b0>)

In [ ]:
train_df = pd.read_table('ratings_train.txt')
test_df = pd.read_table('ratings_test.txt')

In [ ]:
train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [ ]:
train_df['label'].value_counts()

,count
label,
0,75173
1,74827


In [ ]:
print('훈련 데이터 샘플의 개수 : {}'.format(len(train_df)))
print('테스트 데이터 샘플의 개수 : {}'.format(len(test_df)))

훈련 데이터 샘플의 개수 : 150000
테스트 데이터 샘플의 개수 : 50000


In [ ]:
from torchtext import data # torchtext.data 임포트
from konlpy.tag import Okt

In [ ]:
# Okt 토크나이저로 사용
tokenizer = Okt()

In [ ]:
ID = data.Field(sequential = False,
                use_vocab = False) # 실제 사용은 하지 않을 예정

TEXT = data.Field(batch_first = True,
                  fix_length = 30,
                  tokenize=tokenizer.morphs,  # tokenize = Okt()로 한국어 사용 가능
                  pad_token='<pad>',
                  unk_token='<unk>',
                  pad_first=True)

LABEL = data.Field(sequential=False,
                   use_vocab=False,
                   is_target=True)

In [ ]:
from torchtext.data import TabularDataset

In [ ]:
train_data, test_data = TabularDataset.splits(
        path='.', train='ratings_train.txt', test='ratings_test.txt', format='tsv',
        fields=[('id', ID), ('text', TEXT), ('label', LABEL)], skip_header=True)

In [ ]:
print('훈련 샘플의 개수 : {}'.format(len(train_data)))
print('테스트 샘플의 개수 : {}'.format(len(test_data)))

훈련 샘플의 개수 : 150000
테스트 샘플의 개수 : 50000


In [ ]:
print(vars(train_data[0]))

{'id': '9976970', 'text': ['아', '더빙', '..', '진짜', '짜증나네요', '목소리'], 'label': '0'}


In [ ]:
TEXT.build_vocab(train_data, min_freq=10, max_size=10000)
# min_freq : 단어 집합에 추가 시 단어의 최소 등장 빈도 조건을 추가.
# max_size : 단어 집합의 최대 크기를 지정.

In [ ]:
vocab_size = len(TEXT.vocab)
# Vocabulary Info
print(f'Vocab Size : {vocab_size}')

print('Vocab Examples : ')
for idx, (k, v) in enumerate(TEXT.vocab.stoi.items()):
    # stoi로 단어와 각 단어의 정수 인덱스가 저장되어져 있는 딕셔너리 객체에 접근
    if idx >= 10:
        break
    print('\t', k, v)
print('---------------------------------')

Vocab Size : 10002
Vocab Examples : 
	 <unk> 0
	 <pad> 1
	 . 2
	 이 3
	 영화 4
	 의 5
	 .. 6
	 가 7
	 에 8
	 을 9
---------------------------------


In [ ]:
print('단어 집합의 크기 : {}'.format(len(TEXT.vocab)))

단어 집합의 크기 : 10002


In [ ]:
print(TEXT.vocab.stoi)

defaultdict(<bound method Vocab._default_unk_index of <torchtext.vocab.Vocab object at 0x7a752eca5a50>>, {'<unk>': 0, '<pad>': 1, '.': 2, '이': 3, '영화': 4, '의': 5, '..': 6, '가': 7, '에': 8, '을': 9, '...': 10, '도': 11, '들': 12, ',': 13, '는': 14, '를': 15, '은': 16, '너무': 17, '?': 18, '한': 19, '다': 20, '정말': 21, '적': 22, '만': 23, '!': 24, '진짜': 25, '으로': 26, '점': 27, '로': 28, '에서': 29, '연기': 30, '과': 31, '평점': 32, '것': 33, '최고': 34, '내': 35, '~': 36, '나': 37, '그': 38, '잘': 39, '와': 40, '안': 41, '인': 42, '이런': 43, '스토리': 44, '생각': 45, '못': 46, '....': 47, '왜': 48, '드라마': 49, '게': 50, '이다': 51, '감동': 52, '사람': 53, '1': 54, '하는': 55, '보고': 56, '하고': 57, '말': 58, '고': 59, '아': 60, '더': 61, '때': 62, 'ㅋㅋ': 63, '배우': 64, '거': 65, '감독': 66, '그냥': 67, '요': 68, '본': 69, '재미': 70, '내용': 71, '뭐': 72, '중': 73, '까지': 74, '!!': 75, '좀': 76, '쓰레기': 77, '보다': 78, '없는': 79, '시간': 80, '수': 81, '지': 82, '네': 83, '봤는데': 84, '10': 85, '작품': 86, '사랑': 87, '할': 88, '없다': 89, '다시': 90, '하나': 91, '볼': 92, '마지막': 93, 

In [ ]:
import re
import sys
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchtext import data, datasets

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from torchtext.data import Iterator

batch_size = 32
train_loader = Iterator(dataset=train_data, batch_size = batch_size, device=device)
test_loader = Iterator(dataset=test_data, batch_size = batch_size, device=device)

In [ ]:
print('훈련 데이터의 미니 배치 수 : {}'.format(len(train_loader)))
print('테스트 데이터의 미니 배치 수 : {}'.format(len(test_loader)))

훈련 데이터의 미니 배치 수 : 4688
테스트 데이터의 미니 배치 수 : 1563


In [ ]:
hidden_dim = 128
vocab_size = len(TEXT.vocab)
embed_dim = 300
n_classes = 1
n_layers = 1

In [ ]:
class LSTM(nn.Module):
    def __init__(self,n_layers, hidden_dim, vocab_size, embed_dim, n_classes, dropout_p=0.2):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embed_dim)
        self.LSTM = nn.LSTM (input_size = embed_dim,
                           hidden_size = hidden_dim,
                           num_layers = n_layers,
                           dropout = dropout_p,
                           batch_first = True) #
        self.drop = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        emb = self.emb(x)
        # emb : (Batch_Size, Max_Seq_Length, Emb_dim)
        output, hidden = self.LSTM(emb)
        last_output = output[:,-1,:]# (배치 크기, 은닉 상태의 크기)의 텐서로 크기가 변경됨. 즉, 마지막 time-step의 은닉 상태만 가져온다.
        x = self.fc(self.drop(last_output))

        return x

In [ ]:
loss_fn = nn.BCEWithLogitsLoss().to(device)

def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum()/len(correct)
    return acc

In [ ]:
LSTM_model = LSTM(n_layers, hidden_dim, vocab_size, embed_dim, n_classes).to(device)
optimizer = torch.optim.Adam(LSTM_model.parameters(), lr=0.001)

In [ ]:
def train(model, iterator, optimizer, loss_fn, idx_epoch, batch_size):

    epoch_loss = 0
    epoch_acc = 0

    model.train()
    batch_size = batch_size

    for idx, batch in enumerate(iterator):

        # Initializing
        optimizer.zero_grad()
        label = batch.label.float()
        # Forward
        predictions = model(batch.text).squeeze()
        loss = loss_fn(predictions, label)
        acc = binary_accuracy(predictions,label)

        sys.stdout.write(
                    "\r" + f"[Train] Epoch : {idx_epoch:^3}"\
                    f"[{(idx + 1) * batch_size} / {len(iterator) * batch_size} ({100. * (idx + 1) / len(iterator) :.4}%)]"\
                    f"  Loss: {loss.item():.4}"\
                    f"  Acc : {acc.item():.4}"\
                    )

        # Backward
        loss.backward()
        optimizer.step()

        # Update Epoch Performance
        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss/len(iterator) , epoch_acc/len(iterator)

In [ ]:
def evaluate(model, iterator, loss_fn):

    epoch_loss = 0
    epoch_acc = 0

    # evaluation mode
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            label = batch.label.float()
            predictions = model(batch.text).squeeze(1)
            loss = loss_fn(predictions, label)
            acc = binary_accuracy(predictions, label)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
N_EPOCH = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCH):
    train_loss, train_acc = train(LSTM_model, train_loader, optimizer, loss_fn, epoch, batch_size)
    valid_loss, valid_acc = evaluate(LSTM_model, test_loader, loss_fn)
    print('')
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(LSTM_model.state_dict(), f'./LSTM_best_kor.pt')
        print(f'\t Saved at {epoch}-epoch')

    print(f'\t Epoch : {epoch} | Train Loss : {train_loss:.4} | Train Acc : {train_acc:.4}')
    print(f'\t Epoch : {epoch} | Valid Loss : {valid_loss:.4} | Valid Acc : {valid_acc:.4}')

[Train] Epoch :  0 [150016 / 150016 (100.0%)]  Loss: 0.1339  Acc : 0.9375
	 Saved at 0-epoch
	 Epoch : 0 | Train Loss : 0.3982 | Train Acc : 0.8138
	 Epoch : 0 | Valid Loss : 0.3519 | Valid Acc : 0.8427
[Train] Epoch :  1 [150016 / 150016 (100.0%)]  Loss: 0.1633  Acc : 0.9375
	 Saved at 1-epoch
	 Epoch : 1 | Train Loss : 0.2963 | Train Acc : 0.8712
	 Epoch : 1 | Valid Loss : 0.3396 | Valid Acc : 0.8482
[Train] Epoch :  2 [150016 / 150016 (100.0%)]  Loss: 0.1187  Acc : 1.0
	 Epoch : 2 | Train Loss : 0.2418 | Train Acc : 0.8984
	 Epoch : 2 | Valid Loss : 0.3525 | Valid Acc : 0.848
[Train] Epoch :  3 [150016 / 150016 (100.0%)]  Loss: 0.05512  Acc : 1.0
	 Epoch : 3 | Train Loss : 0.1948 | Train Acc : 0.9196
	 Epoch : 3 | Valid Loss : 0.3854 | Valid Acc : 0.8478
[Train] Epoch :  4 [150016 / 150016 (100.0%)]  Loss: 0.09123  Acc : 1.0
	 Epoch : 4 | Train Loss : 0.1531 | Train Acc : 0.938
	 Epoch : 4 | Valid Loss : 0.4209 | Valid Acc : 0.8446
